In [6]:
%pip install xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.7/131.7 MB 86.4 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 342.1/342.1 MB 46.7 MB/s  0:00:06m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [xgboost]m1/2 [xgboost]

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /anaconda/envs/azureml_py310_sdkv2/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
from azure.ai.ml.entities import AzureBlobDatastore
from azure.ai.ml import command, Input, Output, MLClient
from azure.identity import DefaultAzureCredential
import pandas as pd
import os 
import logging
import mlflow
import time
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes, InputOutputModes
import mltable
from mltable import MLTableHeaders, MLTableFileEncoding, DataType

ml_client = MLClient.from_config(credential=DefaultAzureCredential())

/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/azureml/dataprep/api/_loggerfactory.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Found the config file in: /config.json


In [2]:
mlflow_tracking_uri = ml_client.workspaces.get(ml_client.workspace_name).mlflow_tracking_uri

In [3]:
mlflow.set_tracking_uri(mlflow_tracking_uri)

In [2]:
#Most models can be logged with autologging, in some situations it may fail depending on the 
#flavor used in YAML file
import mlflow
from xgboost import XGBClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Load sample data and split into train/test sets.
X, y = load_breast_cancer(return_X_y=True, as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

mlflow.autolog()

model = XGBClassifier(eval_metric="logloss")
model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

2026/09/10 11:44:58 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2026/09/10 11:44:58 INFO mlflow.tracking.fluent: Autologging successfully enabled for xgboost.
2026/09/10 11:45:00 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID 'f8216533-be8f-4f14-97f3-3a986f719380', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current xgboost workflow
Matplotlib is building the font cache; this may take a moment.
2026/09/10 11:45:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/10 11:45:25 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during xgboost autologging: API request to endpoint /api/2.0/mlflow/logged-models failed with error code 404 != 200. Response body: ''


🏃 View run green_head_vsnxtqv4 at: https://centralindia.api.azureml.ms/mlflow/v2.0/subscriptions/5cc08b97-8906-4620-861c-088452556a7b/resourceGroups/demogroup/providers/Microsoft.MachineLearningServices/workspaces/mlw-mlops-3nftht/#/experiments/ffd227fc-eca7-4360-ac28-4cbbee441947/runs/f8216533-be8f-4f14-97f3-3a986f719380
🧪 View experiment at: https://centralindia.api.azureml.ms/mlflow/v2.0/subscriptions/5cc08b97-8906-4620-861c-088452556a7b/resourceGroups/demogroup/providers/Microsoft.MachineLearningServices/workspaces/mlw-mlops-3nftht/#/experiments/ffd227fc-eca7-4360-ac28-4cbbee441947


In [6]:
# The above experiment failed due to dependeny mismtach between mlflow and xgboost
import mlflow
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
from mlflow.models import infer_signature

mlflow.autolog(log_models=False)

with mlflow.start_run():
    model = XGBClassifier(eval_metric="logloss")
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)

    # Infer the signature.
    signature = infer_signature(X_test, y_test)

    # Sample the data.
    input_example = X_train.sample(n=1)

    # Log the model manually.
    mlflow.xgboost.log_model(model, 
                             artifact_path="classifier", 
                             signature=signature,
                             input_example=input_example)

2026/09/10 12:15:32 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2026/09/10 12:15:32 INFO mlflow.tracking.fluent: Autologging successfully enabled for xgboost.
2026/09/10 12:15:43 WARNING mlflow.sklearn: Failed to log evaluation dataset information to MLflow Tracking. Reason: BAD_REQUEST: Response: {'Error': {'Code': 'UserError', 'Severity': None, 'Message': 'Cannot log the same dataset with different context', 'MessageFormat': None, 'MessageParameters': None, 'ReferenceCode': None, 'DetailsUri': None, 'Target': None, 'Details': [], 'InnerError': None, 'DebugInfo': None, 'AdditionalInfo': None}, 'Correlation': {'operation': '2d8f0ef28f638d1cc04eebd89c1a8c1c', 'request': '98b4314f7445a81b'}, 'Environment': 'centralindia', 'Location': 'centralindia', 'Time': '2026-09-10T12:15:43.4640263+00:00', 'ComponentName': 'mlflow', 'statusCode': 400, 'error_code': 'BAD_REQUEST'}


🏃 View run placid_dress_y74x4pv0 at: https://centralindia.api.azureml.ms/mlflow/v2.0/subscriptions/5cc08b97-8906-4620-861c-088452556a7b/resourceGroups/demogroup/providers/Microsoft.MachineLearningServices/workspaces/mlw-mlops-3nftht/#/experiments/ffd227fc-eca7-4360-ac28-4cbbee441947/runs/e73e2bf4-7334-41de-bc8b-f18efcaae787
🧪 View experiment at: https://centralindia.api.azureml.ms/mlflow/v2.0/subscriptions/5cc08b97-8906-4620-861c-088452556a7b/resourceGroups/demogroup/providers/Microsoft.MachineLearningServices/workspaces/mlw-mlops-3nftht/#/experiments/ffd227fc-eca7-4360-ac28-4cbbee441947


MlflowException: API request to endpoint /api/2.0/mlflow/logged-models failed with error code 404 != 200. Response body: ''